# Spectral Analysis — Meta-Analysis of Phases 1-10

**Goal**: Systematically analyze all gathered experimental data to understand which spectral features universalize and why.

**Methodology**:
1. **Load Raw Caches**: Aggregate `.pkl` files from Phases 4, 7, 8, 9, and 10.
2. **Unify Granularity**: Convert all samples to a "Statement-Level" representation for RAG, and "Trace-Level" for Reasoning/QA.
3. **Extract Universal Features**: Run the full 12-feature suite on every trace.
4. **Analyze Importance**: Use Random Forest to rank features across domains (Math, GPQA, GSM8K, QA, RAG).
5. **Map Feature Topology**: Visualize the correlation between spectral features.

## Section 1 — Setup

In [ ]:
import os, sys, shutil, pickle, torch, numpy as np, pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = '/content/hallucination_detection'
BRANCH   = 'research/feature-expansion'

if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {BRANCH} https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q scipy seaborn tqdm scikit-learn')

from spectral_utils import (
    extract_all_features, sw_var_peak_with_window, sw_var_peak_adaptive,
    FEAT_NAMES, zscore, boot_auc, nadler_fuse, simple_average_fusion, best_nadler_on,
    segment_by_citations, lciteeval_grounding_label
)

print('spectral_utils imported OK')

## Section 2 — Data Consolidation

In [ ]:
# Mapping of Phase -> (Directory, Domain, Label_Type)
# Paths fixed to match historical notebook BASE_DIRs
DATA_MAP = {
    'Phase 4/5 (Math)':   ('/content/drive/MyDrive/epr_spectral_phase4', 'Math', 'binary'),
    'Phase 7 (GSM8K)':    ('/content/drive/MyDrive/epr_spectral_gsm8k_vs_lapei', 'GSM8K', 'binary'),
    'Phase 8 (GPQA)':     ('/content/drive/MyDrive/epr_spectral_gpqa_72b', 'GPQA', 'binary'),
    'Phase 9 (QA)':       ('/content/drive/MyDrive/spectral_phase9_cache', 'QA', 'binary'),
    'Phase 10 (RAG)':     ('/content/drive/MyDrive/hallucination_detection/cache/phase10_main/raw', 'RAG', 'statement'),
}

def load_master_dataset():
    all_rows = []
    
    for phase_name, (base_path, domain, ltype) in DATA_MAP.items():
        if not os.path.exists(base_path):
            print(f"Skipping {phase_name}: Path not found at {base_path}")
            continue
            
        # Recursively find all .pkl files
        pkl_files = []
        for root, _, files in os.walk(base_path):
            for f in files:
                if f.endswith('.pkl'):
                    pkl_files.append(os.path.join(root, f))
        
        for fpath in tqdm(pkl_files, desc=f"Loading {phase_name}"):
            fname = os.path.basename(fpath)
            # Skip aggregate result files if they don't contain raw data (optional check)
            if 'summary' in fname or 'final_results' in fname: continue
            
            with open(fpath, 'rb') as f:
                try:
                    data = pickle.load(f)
                except Exception:
                    continue
            
            # Handle list of samples (Standard RAW format)
            if isinstance(data, list):
                for sample in data:
                    if not isinstance(sample, dict) or 'output' not in sample: continue
                    out = sample['output']
                    if 'token_entropies' not in out: continue
                    
                    if ltype == 'statement':
                        # RAG Statement-Level Logic
                        if 'row' not in sample: continue
                        segments = segment_by_citations(out['generated_text'], out['token_offsets'])
                        for seg in segments:
                            label = lciteeval_grounding_label(sample['row'], seg['text'])
                            if label is None: continue
                            h_slice = out['token_entropies'][seg['token_start']:seg['token_end']]
                            if len(h_slice) < 3: continue
                            feats = extract_all_features(h_slice)
                            if feats: all_rows.append({
                                'phase': phase_name, 'domain': domain, 'model': fname.split('__')[0],
                                'label': 1 if label == 'G' else 0, 'h_trace': h_slice, **feats
                            })
                    else:
                        # Reasoning/QA Trace-Level Logic
                        h = out['token_entropies']
                        if len(h) < 3: continue
                        label = sample.get('is_correct', sample.get('label', None))
                        if label is None: continue
                        feats = extract_all_features(h)
                        if feats: all_rows.append({
                            'phase': phase_name, 'domain': domain, 'model': fname.split('_')[0],
                            'label': int(label), 'h_trace': h, **feats
                        })
            
            # Handle processed dict (Result format - useful for global AUCs but not param optimization)
            elif isinstance(data, dict) and 'feat_arrays' in data and 'labels' in data:
                # We can't re-run features on this, but we can use it for Importance Ranking
                # For now, skip to ensure we only have data we can OPTIMIZE.
                pass
                    
    return pd.DataFrame(all_rows)

df = load_master_dataset()
if not df.empty:
    print(f"Master Dataset created: {len(df)} samples")
    print(df.groupby(['domain', 'phase']).size())
else:
    print("Master Dataset is empty. Check paths and Drive mounting.")

## Section 3 — Meta-Analysis

In [ ]:
if not df.empty:
    # 1. Feature Correlation Matrix
    plt.figure(figsize=(12, 10))
    corr = df[FEAT_NAMES].corr(method='spearman')
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title("Spectral Feature Correlation Topology (Global)")
    plt.show()

In [ ]:
if not df.empty:
    # 2. Random Forest Feature Importance per Domain
    unique_domains = df['domain'].unique()
    domain_importances = {}

    for domain in unique_domains:
        sub_df = df[df['domain'] == domain]
        if len(sub_df['label'].unique()) < 2: continue
        
        X = sub_df[FEAT_NAMES]
        y = sub_df['label']
        
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X, y)
        
        domain_importances[domain] = pd.Series(rf.feature_importances_, index=FEAT_NAMES).sort_values(ascending=False)
        
        plt.figure(figsize=(10, 4))
        domain_importances[domain].plot(kind='bar')
        plt.title(f"Feature Importance: {domain}")
        plt.ylabel("Gini Importance")
        plt.show()

In [ ]:
if not df.empty and domain_importances:
    # 3. Identify Universal Features
    rank_df = pd.DataFrame({d: imp.rank(ascending=False) for d, imp in domain_importances.items()})
    rank_df['Average_Rank'] = rank_df.mean(axis=1)
    print("Universal Feature Ranking (Top = Best across all domains):")
    print(rank_df.sort_values('Average_Rank'))

## Section 5 — Spectral Band Optimization

Our current features use a fixed 0.15 normalized frequency cutoff for 'Low' vs 'High' bands. This section empirically optimizes that boundary by sweeping the cutoff and measuring the resulting AUC for 'Band Power Ratio'.

In [ ]:
def compute_band_ratio(ents, cutoff):
    e = np.array(ents, dtype=float); N = len(e)
    if N < 8: return 0.5
    e_ac = e - e.mean()
    fft_vals = np.fft.rfft(e_ac)
    psd = np.abs(fft_vals) ** 2
    freqs = np.fft.rfftfreq(N)
    low_power = psd[freqs <= cutoff].sum()
    high_power = psd[freqs > cutoff].sum()
    return low_power / (high_power + 1e-12)

def optimize_bands_real(sub_df, domain_name):
    cutoffs = np.linspace(0.05, 0.45, 20)
    aucs = []
    
    print(f"Optimizing bands for {domain_name}...")
    for c in tqdm(cutoffs):
        scores = [compute_band_ratio(h, c) for h in sub_df['h_trace']]
        # Note: Orientation might flip, so we take max of AUC and 1-AUC
        a = roc_auc_score(sub_df['label'], scores)
        aucs.append(max(a, 1-a))
    
    plt.figure(figsize=(8, 4))
    plt.plot(cutoffs, aucs, marker='o')
    plt.axvline(0.15, color='red', linestyle='--', label='Current (0.15)')
    plt.title(f"Band Cutoff Sensitivity: {domain_name}")
    plt.xlabel("Normalized Frequency Cutoff")
    plt.ylabel("AUC (Power Ratio)")
    plt.legend()
    plt.show()
    return cutoffs[np.argmax(aucs)], np.max(aucs)

if not df.empty:
    for domain in df['domain'].unique():
        sub = df[df['domain']==domain]
        if len(sub['label'].unique()) < 2: continue
        best_c, best_a = optimize_bands_real(sub, domain)
        print(f"Best Cutoff for {domain}: {best_c:.3f} (AUC: {best_a:.2%})")

## Section 6 — Multi-Parameter Sensitivity Analysis

In [ ]:
def sweep_window_size(sub_df, domain):
    windows = [3, 5, 8, 12, 16, 24, 32]
    aucs = []
    for w in tqdm(windows, desc=f"Window sweep {domain}"):
        scores = [sw_var_peak_with_window(h, w) for h in sub_df['h_trace']]
        a = roc_auc_score(sub_df['label'], scores)
        aucs.append(max(a, 1-a))
    
    plt.figure(figsize=(8, 4))
    plt.plot(windows, aucs, marker='s')
    plt.title(f"Window Size Sensitivity: {domain}")
    plt.xlabel("Window Size (tokens)")
    plt.ylabel("AUC (sw_var_peak)")
    plt.show()

if not df.empty:
    for domain in df['domain'].unique():
        sub = df[df['domain']==domain]
        if len(sub['label'].unique()) < 2: continue
        sweep_window_size(sub, domain)

## Section 7 — Global vs. Local Optima Comparison

In [ ]:
print("Meta-Analysis Ready. Results will inform the implementation of Hurst and Wavelet features.")

## Section 8 — Strategic Conclusions & Next Steps

### Key Findings:
1. **Universal Features**: Which features worked everywhere regardless of parameters?
2. **Optimization Lift**: Does per-domain tuning provide >2% AUC lift? If not, we stay with the 'Universal' setting for simplicity.
3. **Scale Invariance**: Do optimal parameters hold across 7B and 72B models?

### Expansion Roadmap:
1. **Hurst Exponents**: Implement if 'Long-Range Dependency' is identified as a gap in RAG.
2. **Wavelets**: Implement if 'Local Bursts' (STFT/Windowed Var) are the primary signal drivers.
3. **Domain-Adaptive Fusion**: If lift is high, implement a classifier-based 'Gate' that selects the optimal spectral band based on the task type.